# 🤖 Indoor Object Detection: Free Cloud GPU Training (Kaggle)

Train a custom, ground-level **YOLOv8 / YOLOv11** model for autonomous **Indoor Object Detection** using Kaggle's free GPU quota (30 GPU-hours/week).

### ⚙️ One-Time Kaggle Settings (Right Sidebar):
1. **Session Options → Accelerator:** Select **GPU T4 x 2** (or GPU P100).
2. **Session Options → Internet:** Toggle to **ON** (required to download base weights).
3. **Input → Add Input:** Upload your Roboflow exported dataset zip as a Kaggle Dataset and attach it.

In [ ]:
# Step 1: Verify Cloud GPU
!nvidia-smi

In [ ]:
# Step 2: Install Ultralytics & Core Dependencies
!pip install --quiet ultralytics opencv-python onnx

In [ ]:
# Step 3: Copy Unpacked Dataset to Working Directory (Bulletproof)
import os
import glob
import shutil

# 1. Hunt down the yaml file anywhere in the input folder
yaml_files = glob.glob('/kaggle/input/**/data.yaml', recursive=True)

if not yaml_files:
    print("❌ ERROR: Could not find the dataset. Did you attach it on the right sidebar?")
else:
    source_dir = os.path.dirname(yaml_files[0])
    print(f"📦 Found dataset hidden at: {source_dir}")
    
    # 2. Copy everything safely to the writable working directory
    shutil.copytree(source_dir, '/kaggle/working/', dirs_exist_ok=True)
    print("✅ Dataset successfully copied to /kaggle/working/!")
    # 3. Rewrite data.yaml to enforce absolute Kaggle path (Bulletproof)
    yaml_path = '/kaggle/working/data.yaml'
    with open(yaml_path, 'r') as f:
        content = f.read()
    import re
    if 'path:' in content:
        content = re.sub(r'path:.*', 'path: /kaggle/working', content)
    else:
        content = 'path: /kaggle/working\n' + content
    with open(yaml_path, 'w') as f:
        f.write(content)
    print('✅ data.yaml path rewritten for Kaggle!')


In [ ]:
# Step 4: Fine-Tune YOLO on Cloud GPU (Indoor Detection)
from ultralytics import YOLO

# Load Nano base model (ultra-fast for edge devices)
model = YOLO('yolov8n.pt')

# Train for 100 epochs on GPU
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=100,
    imgsz=416,
    batch=32,      # Pushed to 32 to utilize the T4 GPUs
    device=0,
    optimizer='AdamW',
    save=True,
    project='/kaggle/working/runs',
    name='indoor_detection_v1'
)
print('🎉 Training Complete!')

In [ ]:
# Step 5: Export to ONNX for Low-Latency Deployment
best_weights = '/kaggle/working/runs/indoor_detection_v1/weights/best.pt'
trained_model = YOLO(best_weights)
trained_model.export(format='onnx', imgsz=416, simplify=True)
print('✅ Exported to ONNX: best.onnx')

In [ ]:
# Step 6: Package Model Weights for Download
import shutil

# This will zip EVERYTHING inside the runs folder!
shutil.make_archive('/kaggle/working/indoor_detection_results', 'zip', '/kaggle/working/runs')

print('✅ Output Zip Created: /kaggle/working/indoor_detection_results.zip')